# Base unificada por establecimiento — SIMCE 2° medio 2015–2025

Produce la tabla final del Capítulo 3: **una fila por establecimiento (RBD)**, con las medidas anuales desplegadas en columnas `<variable>_<año>`, las categóricas convertidas a texto, y la **georreferenciación** (latitud/longitud) del Directorio Oficial.

**Requisito:** haber ejecutado `01_construccion_panel.ipynb`, que genera `Dataset/Unificados/simce_rbd_unificado_2015_2025.csv` ya cruzado con el Directorio.

---
### Correcciones aplicadas (verificadas contra los datos reales)

| # | Hallazgo en los archivos | Corrección |
|---|---|---|
| 1 | El formato de las categóricas cambia: **texto** hasta 2017, **código** desde 2018, y con **tres vocabularios distintos** | Tabla de equivalencias explícita (el emparejamiento por similitud falla: *"Corporación Municipal"* se parece más a *"Corporación de administración delegada"*) |
| 2 | En `marcadif_*` los códigos **1 y 2 se permutan en 2023**, y además **se fusionan dos categorías** | Traducción con el catálogo **del año de cada dato** + capa canónica armonizada |
| 3 | `cod_depe1` pasa de 5 a 6 categorías en 2018 (aparece *Servicio Local de Educación*) | Catálogo por año; la categoría nueva no se fuerza hacia atrás |
| 4 | En el SIMCE **2015** las columnas `cod_com_rbd`, `cod_pro_rbd` y `cod_deprov_rbd` traen **el mismo valor** (índice de DEPROV, 54 valores) en vez del código de comuna; en 2018 los nombres vienen truncados | Región y comuna se toman **del Directorio**, no del archivo SIMCE |
| 5 | El indicador de cambio de atributo daba 2.919 colegios (artefacto del paso texto→código) | El cambio se evalúa sobre la **categoría canónica**: 384 cambios reales de dependencia |
| 6 | `noaplica = 8` (2024) y `marcapalu_*` (2015) no están en ningún glosario | Se etiquetan como *sin documentar* / concepto propio, sin inventar significado |

In [1]:
import sys
from pathlib import Path

for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]
candidatos = [Path.cwd() / "simce_panel" / "src", Path.cwd().parent / "src", Path.cwd() / "src"]
RUTA_SRC = next((p for p in candidatos if (p / "simce_panel" / "__init__.py").exists()), None)
if RUTA_SRC is None:
    raise FileNotFoundError("No encontré 'src/simce_panel'. Revisa dónde está la carpeta del proyecto.")
sys.path.insert(0, str(RUTA_SRC))

import pandas as pd
import simce_panel as sp
sp.configurar_logging()
print("simce_panel", sp.__version__, "|", sp.__file__)

simce_panel 1.1.0 | <ruta local>


In [2]:
BASE_DIR = "."          # el notebook está junto a la carpeta 'Dataset'
cfg = sp.Config.desde_base(BASE_DIR)
cfg.validar()

# Panel largo enriquecido que produjo el notebook 01 (incluye lat/long).
largo = sp.io_datos.leer_csv(cfg.ruta_salida(sp.config.NOMBRES_SALIDA["rbd"]))
print("Panel largo:", largo.shape)

tiene_geo = {"latitud", "longitud"}.issubset(largo.columns)
print("¿Trae coordenadas?", tiene_geo)
if tiene_geo:
    print("Cobertura de coordenadas: "
          f"{largo['latitud'].notna().mean():.1%} de las filas")

Panel largo: (23605, 65)
¿Trae coordenadas? True
Cobertura de coordenadas: 99.6% de las filas


## 1. Georreferenciación y territorio desde el Directorio

Si el panel aún no trae coordenadas, se cruzan aquí. Además se anexan **región y comuna desde el Directorio** (`region_dir`, `comuna_dir`): los códigos territoriales del propio archivo SIMCE **no son utilizables en 2015** (las tres columnas territoriales repiten el mismo índice de DEPROV) y los nombres de comuna de 2018 vienen truncados.

In [3]:
# Coordenadas (solo si faltaran).
if not tiene_geo:
    largo = sp.directorio.enriquecer_rbd(largo, cfg, guardar=False)
    print("Coordenadas anexadas:", largo["latitud"].notna().mean().round(3))

# Región y comuna desde el Directorio, en columnas nuevas '_dir'.
largo = sp.directorio.anexar_territorio(largo, cfg)
print("\nColumnas territoriales añadidas:",
      [c for c in largo.columns if c.endswith("_dir")])

15:14:11 | INFO    | simce_panel.directorio | Directorio 2015: 15914 RBD con geo.


15:14:11 | INFO    | simce_panel.directorio | Directorio 2016: 15990 RBD con geo.


15:14:12 | INFO    | simce_panel.directorio | Directorio 2017: 16024 RBD con geo.


15:14:12 | WARNING | simce_panel.directorio | Se repararon 1 coordenadas mal formateadas.


15:14:12 | WARNING | simce_panel.directorio | 5 filas tenían latitud y longitud invertidas; se corrigieron.


...

15:14:12 | WARNING | simce_panel.directorio | Directorio 2025: 1 coordenadas requirieron atención (0 reparadas, 1 invertidas, 0 irreparables).


15:14:13 | INFO    | simce_panel.directorio | Directorio 2025: 16768 RBD con geo.


15:14:13 | INFO    | simce_panel.directorio | Territorio 'region' desde Directorio: 100.0% de cobertura.


15:14:13 | INFO    | simce_panel.directorio | Territorio 'comuna' desde Directorio: 100.0% de cobertura.



Columnas territoriales añadidas: ['region_dir', 'comuna_dir']


In [4]:
# Evidencia del problema en los códigos territoriales del SIMCE:
# 2015 tiene muy pocos valores distintos de 'cod_com_rbd' (deberían ser ~330).
if "cod_com_rbd" in largo.columns:
    chk = (largo.groupby("año")["cod_com_rbd"]
           .nunique().rename("comunas_distintas_SIMCE").to_frame())
    if "comuna_dir" in largo.columns:
        chk["comunas_distintas_DIRECTORIO"] = largo.groupby("año")["comuna_dir"].nunique()
    display(chk)

,comunas_distintas_SIMCE,comunas_distintas_DIRECTORIO
año,,
2015,54,328
2016,328,328
2017,330,330
2018,330,330
2022,332,332
2023,332,332
2024,335,335
2025,335,335


## 2. Catálogos de códigos, leídos de los glosarios oficiales

Se extraen las tablas *Valor / Descripción* de cada `*_GLOSAS_*.xlsx`, indexadas por `(concepto, año)`. Los glosarios de 2015–2017 no las traen; para esos años se usa un catálogo supuesto que **se verifica contra los datos** más abajo.

In [5]:
catalogos = sp.glosarios.construir_catalogos(cfg.simce_dir)
cambios = sp.glosarios.detectar_cambios_de_catalogo(catalogos)
print(cambios.tipo_cambio.value_counts().to_dict())
cambios[cambios.tipo_cambio.isin(["PERMUTACIÓN DE CÓDIGOS", "código nuevo",
                                  "código eliminado"])]

15:14:13 | INFO    | simce_panel.glosarios | Glosario 2015: catálogos leídos -> (ninguno)


15:14:13 | INFO    | simce_panel.glosarios | Glosario 2016: catálogos leídos -> (ninguno)


15:14:13 | INFO    | simce_panel.glosarios | Glosario 2017: catálogos leídos -> (ninguno)


15:14:13 | INFO    | simce_panel.glosarios | Glosario 2018: catálogos leídos -> ['marca', 'marcadif', 'noaplica']


15:14:13 | INFO    | simce_panel.glosarios | Glosario 2022: catálogos leídos -> ['dependencia1', 'dependencia2', 'gse', 'marca', 'marcadif', 'noaplica', 'ruralidad', 'significancia']


...

15:14:13 | WARNING | simce_panel.glosarios | Año 2018: sin tabla oficial para 'gse'; se usa catálogo SUPUESTO (verificar contra los datos).


15:14:13 | WARNING | simce_panel.glosarios | Año 2018: sin tabla oficial para 'ruralidad'; se usa catálogo SUPUESTO (verificar contra los datos).


15:14:13 | WARNING | simce_panel.glosarios | Año 2018: sin tabla oficial para 'significancia'; se usa catálogo SUPUESTO (verificar contra los datos).


{'CAMBIO DE SIGNIFICADO': 8, 'cambio de redacción': 4, 'código nuevo': 3, 'PERMUTACIÓN DE CÓDIGOS': 1, 'código eliminado': 1}


,concepto,codigo,año_desde,etiqueta_anterior,etiqueta_nueva,similitud,tipo_cambio
0,dependencia1,6,2022,NaN,Servicio Local de Educación,NaN,código nuevo
1,dependencia2,4,2022,NaN,Servicio Local de Educación,NaN,código nuevo
7,marcadif,2,2023,"No es posible reportar esta información, porqu...","No es posible comparar resultados, porque la c...",0.099,PERMUTACIÓN DE CÓDIGOS
8,marcadif,3,2023,"La comparación no es representativa, porque en...",NaN,NaN,código eliminado
16,noaplica,7,2022,NaN,Comunidad educativa se niega a rendir,NaN,código nuevo


## 3. Estandarización en dos capas

- **`<variable>_desc`** — etiqueta **oficial del año**. Fiel a la fuente; para reportar y auditar.
- **`<variable>_cat`** — **categoría canónica armonizada**, comparable en toda la serie. Es la que se usa para modelar.

La segunda capa es indispensable porque en 2023 la Agencia **fusionó** dos categorías de `marcadif_*`; ninguna traducción código→texto puede deshacer esa fusión.

In [6]:
# Capa 1: etiqueta oficial del año.
estandarizado, rep_desc = sp.estandarizacion.estandarizar_categoricas(
    largo, catalogos, col_anio="año", reemplazar=False)

# Capa 2: categoría canónica armonizada (resuelve códigos y los 3 vocabularios).
estandarizado, rep_cat = sp.estandarizacion.estandarizar_canonico(
    estandarizado, col_anio="año")

print("Sin categoría canónica:", int(rep_cat.n_pendiente.sum()))
rep_cat.groupby("variable")[["n_con_valor", "n_resuelto", "n_pendiente"]].sum()

15:14:22 | WARNING | simce_panel.estandarizacion | Sin catálogo para 'marcapalu' en 2015; 2876 filas sin traducir.


15:14:22 | WARNING | simce_panel.estandarizacion | Sin catálogo para 'marcapalu' en 2016; 2902 filas sin traducir.


15:14:22 | WARNING | simce_panel.estandarizacion | Sin catálogo para 'marcapalu' en 2017; 2923 filas sin traducir.


15:14:22 | WARNING | simce_panel.estandarizacion | Sin catálogo para 'marcapalu' en 2018; 2935 filas sin traducir.


15:14:22 | WARNING | simce_panel.estandarizacion | Sin catálogo para 'marcapalu' en 2022; 2976 filas sin traducir.


...

15:14:22 | WARNING | simce_panel.estandarizacion | Sin catálogo para 'noaplica' en 2024; 3000 filas sin traducir.


15:14:22 | WARNING | simce_panel.estandarizacion | Sin catálogo para 'noaplica' en 2025; 3002 filas sin traducir.


15:14:23 | WARNING | simce_panel.estandarizacion | Quedaron 113905 valores sin resolver; revisa el reporte.


Sin categoría canónica: 0


,n_con_valor,n_resuelto,n_pendiente
variable,,,
cod_depe1,23605,23605,0
cod_depe2,23605,23605,0
cod_grupo,23537,23537,0
cod_rural_rbd,23605,23605,0
marca_lect2m_rbd,623,623,0
marca_mate2m_rbd,694,694,0
marca_nat2m_rbd,106,106,0
marca_soc2m_rbd,105,105,0
marcadif_lect2m_rbd,1875,1875,0


In [7]:
# Comprobación de la armonización: el MISMO código significa cosas distintas
# según el año; la categoría canónica lo corrige.
chk = (estandarizado[estandarizado.marcadif_mate2m_rbd.notna()]
       .groupby(["año", "marcadif_mate2m_rbd"])["marcadif_mate2m_rbd_cat"]
       .agg(lambda s: s.dropna().iloc[0] if len(s.dropna()) else None)
       .reset_index())
chk[chk["año"].isin([2015, 2018, 2022, 2023, 2025])]

,año,marcadif_mate2m_rbd,marcadif_mate2m_rbd_cat
0,2015,Alumnos insuficientes,Estudiantes insuficientes
1,2015,Evaluación actual no permite comparar,Comparación no válida
2,2015,Evaluación anterior no permite comparar,Comparación no válida
9,2018,1.0,Estudiantes insuficientes
10,2018,3.0,Comparación no válida
11,2022,1.0,Estudiantes insuficientes
12,2022,3.0,Comparación no válida
13,2023,1.0,Comparación no válida
14,2023,2.0,Estudiantes insuficientes
17,2025,1.0,Comparación no válida


In [8]:
# Verificación de los catálogos supuestos (2015-2018):
# 'observados_fuera_de_catalogo' debe venir vacío.
val = sp.estandarizacion.validar_catalogos_contra_datos(estandarizado, catalogos)
val[(val["año"] <= 2018) &
    (val.variable.isin(["cod_depe1", "cod_depe2", "cod_grupo", "cod_rural_rbd"]))][
    ["variable", "año", "codigos_observados", "observados_fuera_de_catalogo"]]

,variable,año,codigos_observados,observados_fuera_de_catalogo
0,cod_depe1,2015,[],[]
1,cod_depe1,2016,[],[]
2,cod_depe1,2017,[],[]
3,cod_depe1,2018,"[1, 2, 3, 4, 5, 6]",[6]
8,cod_depe2,2015,[],[]
9,cod_depe2,2016,[],[]
10,cod_depe2,2017,[],[]
11,cod_depe2,2018,"[1, 2, 3, 4]",[4]
16,cod_grupo,2015,[],[]
17,cod_grupo,2016,[],[]


## 4. Tipado numérico

Los puntajes, diferencias y conteos se convierten a numérico antes de pivotar (toleran coma decimal). Si ya aplicaste el saneamiento del notebook 02, los sentinelas y los resultados no representativos ya están en `NaN`.

In [9]:
for c in estandarizado.columns:
    if c.startswith(("prom_", "dif_", "difgru_", "nalu_", "palu_")):
        estandarizado[c] = sp.io_datos.a_numerico(estandarizado[c])
print("Columnas numéricas:",
      sum(str(estandarizado[c].dtype).startswith("float") for c in estandarizado.columns))

Columnas numéricas: 25


## 5. Panel ancho: una fila por establecimiento

- **Identidad**: `rbd`.
- **Atributos de ficha** (nombre, dependencia, GSE, ruralidad, coordenadas, territorio): un valor por colegio, el del **año más reciente**, más `<atributo>_cambio`.
- **Medidas anuales**: `<variable>_<año>`.

El indicador `_cambio` se evalúa sobre la **columna canónica**, no sobre el valor crudo: de lo contrario el paso de texto (≤2017) a código (≥2018) se contaría como cambio en casi todos los colegios.

In [10]:
ancho, mapa_columnas = sp.panel_ancho.construir_panel_ancho(
    estandarizado, col_rbd="rbd", col_anio="año",
    politica_atributos="ultimo", marcar_cambios=True)

print("PANEL ANCHO:", ancho.shape)
print("Roles:", mapa_columnas.rol.value_counts().to_dict())
print("¿Con coordenadas?", {"latitud", "longitud"}.issubset(ancho.columns))
ancho[[c for c in ["rbd", "nom_rbd", "cod_depe1_cat", "cod_grupo_cat",
                    "latitud", "longitud", "n_anios_presente"]
       if c in ancho.columns]].head()

15:14:25 | INFO    | simce_panel.panel_ancho | Columnas -> atributos: 27 | medidas: 81 | descartadas: 3


15:14:38 | INFO    | simce_panel.panel_ancho | Panel ancho: 3160 establecimientos x 704 columnas


PANEL ANCHO: (3160, 704)
Roles: {'medida anual': 648, 'atributo de ficha': 27, 'indicador de cambio': 27, 'identidad': 1, 'derivada': 1}
¿Con coordenadas? True


[Salida omitida en la versión pública: mostraba nombres de establecimientos.]


In [11]:
# Cambios REALES de atributo durante la serie.
cols_cambio = [c for c in ancho.columns if c.endswith("_cambio")]
r = ancho[cols_cambio].sum().sort_values(ascending=False)
r[r > 0].rename("establecimientos_que_cambiaron").to_frame()

,establecimientos_que_cambiaron
nom_reg_rbd_cambio,2977
nom_com_rbd_cambio,2932
nom_deprov_rbd_cambio,2915
cod_pro_rbd_cambio,2906
cod_com_rbd_cambio,2860
cod_deprov_rbd_cambio,2755
cod_grupo_cambio,1662
cod_grupo_cat_cambio,1662
cod_grupo_desc_cambio,1662
nom_pro_rbd_cambio,1205


In [12]:
# Cobertura: en cuántos años aparece cada establecimiento.
print(ancho.n_anios_presente.value_counts().sort_index().to_string())
if "latitud" in ancho.columns:
    print("\nEstablecimientos georreferenciados: "
          f"{ancho['latitud'].notna().sum()} de {len(ancho)} "
          f"({ancho['latitud'].notna().mean():.1%})")

n_anios_presente
1      51
2      34
3      36
4     171
5      37
6      38
7      63
8    2730

Establecimientos georreferenciados: 3158 de 3160 (99.9%)


## 6. Guardado y glosario consolidado

Se guardan dos versiones: la **completa** (todas las variables y las tres capas de cada categórica) y una **reducida** para modelar, solo con Lectura y Matemática y la capa canónica.

In [13]:
ruta_ancho = cfg.ruta_salida("simce_rbd_ancho_2015_2025.csv")
sp.io_datos.escribir_csv(ancho, ruta_ancho)
print("Base completa:", ruta_ancho, ancho.shape)

Base completa: Dataset\Unificados\simce_rbd_ancho_2015_2025.csv (3160, 704)


In [14]:
# Versión reducida para modelamiento.
patrones_medida = ("prom_lect", "prom_mate", "nalu_lect", "nalu_mate")
cols_ficha = [c for c in ["rbd", "nom_rbd", "latitud", "longitud",
                          "region_dir", "comuna_dir",
                          "cod_depe1_cat", "cod_depe2_cat",
                          "cod_grupo_cat", "cod_rural_rbd_cat",
                          "cod_depe1_cambio", "cod_grupo_cambio",
                          "n_anios_presente"] if c in ancho.columns]
cols_medida = [c for c in ancho.columns if c.startswith(patrones_medida)]
reducida = ancho[cols_ficha + cols_medida]

ruta_red = cfg.ruta_salida("simce_rbd_ancho_reducido_2015_2025.csv")
sp.io_datos.escribir_csv(reducida, ruta_red)
print("Base reducida:", ruta_red, reducida.shape)
reducida.head(3)

Base reducida: Dataset\Unificados\simce_rbd_ancho_reducido_2015_2025.csv (3160, 45)


[Salida omitida en la versión pública: mostraba nombres de establecimientos.]


In [15]:
# Glosario consolidado (4 hojas: DICCIONARIO, VALORES, CAMBIOS, RESUMEN).
descripciones = {}
for anio, xlsx in sorted(sp.glosarios.mapear_glosarios(cfg.simce_dir).items()):
    for var, desc in sp.glosarios.leer_descripciones(xlsx).items():
        descripciones.setdefault(var, desc)

ruta_glos = sp.glosario_unificado.exportar_glosario_unificado(
    ancho, mapa_columnas, descripciones, catalogos,
    cfg.ruta_salida("glosario_consolidado_ancho.xlsx"))
print("Glosario:", ruta_glos)

diccionario = sp.glosario_unificado.construir_diccionario(
    ancho, mapa_columnas, descripciones, catalogos)
diccionario.head(12)

15:14:39 | INFO    | simce_panel.glosario_unificado | Glosario consolidado exportado: Dataset\Unificados\glosario_consolidado_ancho.xlsx


Glosario: Dataset\Unificados\glosario_consolidado_ancho.xlsx


[Salida omitida en la versión pública: mostraba nombres de establecimientos.]


---
### Salidas en `Dataset/Unificados/`

| Archivo | Contenido |
|---|---|
| `simce_rbd_ancho_2015_2025.csv` | Base completa: una fila por RBD, con coordenadas |
| `simce_rbd_ancho_reducido_2015_2025.csv` | Versión para modelar: Lectura y Matemática, capa canónica |
| `glosario_consolidado_ancho.xlsx` | Diccionario + catálogo de valores + auditoría de rupturas |

**Para el análisis espacial:** usa la base reducida; ya trae `latitud`/`longitud` (una por colegio), la dependencia y el GSE armonizados, y `cod_depe1_cambio` para identificar los colegios traspasados a un SLEP durante la serie.

In [16]:
sp.directorio.diagnostico_formato_coordenadas(cfg, [2015, 2016, 2017])

,año,col_lat,col_lon,n_filas,vacios,con_coma_decimal,con_punto_decimal,sin_separador,no_convertibles,ejemplos_problematicos
0,2015,latitud,longitud,15914,0,0,12534,3380,3379,"['0' / '0', '0' / '0', '0' / '0']"
1,2016,latitud,longitud,15990,0,0,12534,3456,3455,"['0' / '0', '0' / '0', '0' / '0']"
2,2017,LATITUD,LONGITUD,16024,3488,12535,0,1,0,[]


In [17]:
sin_geo = ancho[ancho["latitud"].isna()]
sin_geo[["rbd", "nom_rbd", "region_dir", "n_anios_presente"]]

[Salida omitida en la versión pública: mostraba nombres de establecimientos.]
